In [2]:
from utils.stat_uitils import load_judge_results, calculate_statistics, print_statistics

In [ ]:
# Smoke-test session produced by scripts/eval.sh edit (sp + mp)
# Layout: results_smoke/<task>/<session>/<page_category>/<id>/judge.json
BASE_DIR = "./results_smoke"
TASK = "edit"
SESSION = "claude-opus-4-6-20260205_text_20260418_164548"

SESSION_PATH = f"{BASE_DIR}/{TASK}/{SESSION}"

TASK_TYPES = [TASK]


In [ ]:
all_stats = {}

for task_type in TASK_TYPES:
    sp_results = load_judge_results(SESSION_PATH, task_type, page_category="sp")
    mp_results = load_judge_results(SESSION_PATH, task_type, page_category="mp")
    results = sp_results + mp_results
    if not results:
        print(f"No judge results found for {task_type}")
        continue

    stats = calculate_statistics(results, task_type)
    all_stats[task_type] = stats

    print_statistics(stats, task_type)


if len(all_stats) > 1:
    print(f"\n{'='*80}")
    print("CROSS-TASK SUMMARY")
    print(f"{'='*80}")
    for task_type, stats in all_stats.items():
        print(f"\n{task_type.upper()}:")
        print(f"  Total Folders: {stats['total_folders']}")
        print(f"  Overall Harmonic Mean: {stats['overall_harmonic_mean'] * 10:.4f}")

    print(f"\n[Worst Metrics]")
    print(f"Task\tWorst-of-1\tWorst-of-5\tWorst Half Tasks")
    for task_type, stats in all_stats.items():
        print(f"{task_type.upper()}\t{stats['overall_worst_of_1'] * 10:.2f}\t{stats['overall_worst_of_5'] * 10:.2f}\t{stats['overall_worst_half_tasks'] * 10:.2f}")

    print(f"\n[Dimension Harmonic Means]")
    for task_type, stats in all_stats.items():
        parts = [
            f"{d.replace('_', ' ').title()}={stats[f'overall_{d}_harmonic'] * 10:.2f}"
            for d in stats['dim_names']
        ]
        print(f"{task_type.upper()}\t" + "\t".join(parts))


In [ ]:
for task_type, stats in all_stats.items():
    print(f"\n📊 Task Type: {task_type}")
    for page_cat in ["mp", "sp"]:
        cat_items = sorted(
            [(folder, data) for folder, data in stats["folder_scores"].items()
             if folder.startswith(page_cat + "_")]
        )
        top10 = cat_items[:10]
        for folder, data in top10:
            print(f"{data['harmonic_mean']:.4f}")
